# 03 · Model audit: what was the score optimizing?

The model looked strong, but two separate validity questions emerged: (1) how much signal was already-underway deterioration, and (2) whether runoff-event frequency aligned with dollar exposure.

In [ ]:
from deposit_runoff.config import load_config
from deposit_runoff.demo_data import make_demo_daily_panel
from deposit_runoff.features import build_v1_dataset

cfg = load_config('../config/public.yaml')
d = cfg['data']
daily = make_demo_daily_panel(n_accounts=d['n_accounts'], start_date=str(d['start_date']), end_date=str(d['end_date']), seed=cfg['seed'])
snapshots = [*map(str, cfg['snapshots']['train']), str(cfg['snapshots']['validation']), str(cfg['snapshots']['test'])]
v1 = build_v1_dataset(daily, snapshots, **cfg['v1'])

In [ ]:
from deposit_runoff.modeling import prepare_model_frame, train_lightgbm
from deposit_runoff.split import chronological_split

v1 = prepare_model_frame(v1)
train, val, test = chronological_split(v1, train_dates=cfg['snapshots']['train'], validation_date=cfg['snapshots']['validation'], test_date=cfg['snapshots']['test'])
model = train_lightgbm(train, val, cfg['model'])
test_pred = model.predict_proba(test[model.feature_name_])[:, 1]

In [ ]:
from deposit_runoff.metrics import balance_extreme_capture, top_fraction_metrics

model_top10 = top_fraction_metrics(test, test_pred, fraction=0.10)
large_balance = balance_extreme_capture(test, largest_first=True, fraction=0.10)
small_balance = balance_extreme_capture(test, largest_first=False, fraction=0.10)
model_top10, large_balance, small_balance

## Verified objective-audit finding, sanitized

| Top 10% ranked by | Event capture | Dollar-decline capture |
|---|---:|---:|
| Lowest balances | ~13% | <0.03% |
| Highest balances | ~6% | ~78% |
| LightGBM risk | ~56% | ~50% |

The key conclusion is not that one ranking is universally better. They optimize **different business objectives**.

In [ ]:
from deposit_runoff.audit import deterioration_sensitivity

deterioration_sensitivity(test, test_pred)

## Feature attribution

The internal analysis used SHAP on a test sample and found recent balance deterioration to be the strongest signal. This repo keeps SHAP as an optional analysis dependency, but does not publish the institution-specific feature-attribution output.

In [ ]:
import shap

sample = test[model.feature_name_].sample(min(2000, len(test)), random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)
# shap.summary_plot(shap_values[-1] if isinstance(shap_values, list) else shap_values, sample)